# 第 8 章习题与解答

## Exercise 8.1

**题目**:为什么 `PretrainDataset` 把 pad 位置的 label 设为 -100?如果不设会怎样?

<details><summary><b>参考答案</b></summary>

`F.cross_entropy(logits, labels, ignore_index=-100)` 会跳过 label=-100 的位置。

**如果不设(-100 改成 pad_token_id)**:
- 模型会被要求「在 pad 位置也预测正确的下一个 token」
- 但 pad 后面还是 pad,模型会学着「生成大量 pad token」
- 这不是我们想要的行为 —— 浪费学习能力在无意义的 padding 上

设 -100 后,pad 位置的 loss 被忽略,模型只学习真实文本的 token 预测。

</details>

## Exercise 8.2

**题目**:画出余弦 LR 曲线的大致形状,标注 warmup 底部(10%)和峰值。为什么不在 step=0 直接使用峰值 lr?

<details><summary><b>参考答案</b></summary>

余弦 LR 曲线形状:

```
lr
1.0 ────╮ 峰值(step≈0)
         ╲
0.55 ─────╲────────────(step=T/2)
           ╲
0.1 ────────╲────────── 底部(step=T)
```

**为什么不在 step=0 用峰值?** 实际上 minimind 的公式在 step=0 就接近峰值(0.1+0.45×2=1.0)。但通常的 warmup 策略(如 GPT-3)会在前几千步从 0 线性升到峰值,原因:

1. **初始权重是随机的**:如果一开始就用大 lr,梯度方向噪声极大,可能把权重推向不可恢复的坏区域
2. **Adam 的动量尚未稳定**:优化器的前几步没有历史梯度信息,大步容易跑偏

minimind 简化了 warmup(底部 10% 而非 0),因为模型小(64M),不太需要谨慎的 warmup。

</details>

## Exercise 8.3

**题目**:如果训练中途 GPU 数量从 4 变成 2(比如一块卡坏了),DDP checkpoint 如何正确恢复?

<details><summary><b>参考答案</b></summary>

DDP checkpoint 包含:
- `model.state_dict()`:权重(与 GPU 数量无关)
- `optimizer.state_dict()`:Adam 的一阶/二阶动量(与 GPU 数量无关)
- `epoch` 和 `global_step`:进度

**恢复时**:
1. 加载模型和优化器状态(完全兼容,因为 state_dict 不依赖 GPU 数量)
2. 用新的 GPU 数量(2)初始化 DDP
3. `SkipBatchSampler` 根据新的 batch_size(=2 卡 × batch_size_per_gpu)跳过已训练的 batch

**关键**:DDP 的梯度同步是 all-reduce(取平均),所以 4 卡和 2 卡的每步梯度更新方向相同(只是 4 卡处理的 batch 数是 2 卡的 2 倍)。checkpoint 记录的是 global_step(总 batch 数),恢复时按新配置继续即可。

</details>